In [1]:
import json
import Levenshtein
import pandas as pd

In [2]:
batches_direct_spelling = ['10003120', '10003940', '10004561', '10004621', '10004652', '10004821', '10005454', '10007783', '10013259'] + \
    ['10014762', '10018083', '10018098', '10018103', '10018126', '10018175', '10022891', '10008732', '10009435', '10009439', '10009451', '10014232', '10016113', '10018590']

In [3]:
df_lm = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/LM_2008.txt', sep=';', encoding='latin-1', dtype=str, index_col=[3,4,5,6])

In [4]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_link_epoch3.json', 'r') as f:
    sc_dict = json.load(f)

In [5]:
gkomm_dict = {}

for batch in sc_dict.keys():
    gkomms = []
    
    for page in sc_dict[batch].keys():

        if sc_dict[batch][page] is None:
            continue

        for inst in sc_dict[batch][page]:
            #gkomm = inst['gts'][0].split(';')[0]
            gkomms_page = [x[0].split(';')[0] for x in inst['gts']]
            for gk in gkomms_page:
                if gk not in gkomms:
                    gkomms.append(gk)
            
            #if gkomm not in gkomms:
                #gkomms.append(gkomm)

    gkomm_dict[batch] = gkomms

In [7]:
#create spelling dicts for each batch on direct links

spelling_dict = {}

for batch in batches_direct_spelling:

    spelling_dict[batch] = {}

    trakt_allowed = []
    block_allowed = []
    enhet_allowed = []

    for page in sc_dict[batch].keys():

        if sc_dict[batch][page] is not None and sc_dict[batch][page][0]['det_prob'] is not None:

            for inst in sc_dict[batch][page]:

                if len(inst['pred_links']) >= 1 and float(inst['pred_conf']) > 0.95:
                    trakt_allowed.append(inst['pred'].split(';')[0].upper())
                    block_allowed.append(inst['pred'].split(';')[1].upper())
                    enhet_allowed.append(inst['pred'].split(';')[2].upper())

    spelling_dict[batch]['trakt'] = set(trakt_allowed)
    spelling_dict[batch]['block'] = set(block_allowed)
    spelling_dict[batch]['enhet'] = set(enhet_allowed)

In [8]:
#create spelling dict from NAD


df1 = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/batcher_data.txt', '\t', dtype=str, encoding='latin-1')
df2 = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/batcher_data2.txt', '\t', dtype=str, encoding='latin-1')
frames = [df1, df2]
df = pd.concat(frames)

batches = df.groupby('batch')

list_of_batches = df.batch.unique()

for batch in list_of_batches:
    spelling_dict[batch] = {}
    
    df_batch = batches.get_group(batch)
    spelling_dict[batch]['trakt'] = df_batch.GTRAKT.unique()
    spelling_dict[batch]['block'] = df_batch.GBLOCK.unique()
    spelling_dict[batch]['enhet'] = df_batch.GENHET.unique()

In [10]:
def jaro(allowed, pred):
    jaro_list = []
    
    for inst in allowed:
        val = Levenshtein.ratio(pred, inst)
        jaro_list.append((inst, val))

    jaro_list = sorted(jaro_list, key=lambda x: x[1], reverse=True)

    return jaro_list

In [11]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_log_from_sc_link.txt', 'w') as f:

    for i, batch in enumerate(sc_dict.keys()):
        
        no_of_misses = 0
        no_of_corrected = 0

        try:
            gkomm_list = gkomm_dict[batch]
        except:
            print('error')

        for page in sc_dict[batch].keys():

            if sc_dict[batch][page] is not None and sc_dict[batch][page][0]['det_prob'] is not None:

                for inst in sc_dict[batch][page]:

                    inst['sc_links'] = []

                    if len(inst['pred_links']) == 0 and float(inst['pred_conf']) > 0.88:

                        no_of_misses +=1

                        try:
                            trakt = inst['pred'].split(';')[0].upper()
                            block = inst['pred'].split(';')[1].upper()
                            enhet = inst['pred'].split(';')[2].upper()
                        except:
                            continue

                        trakt_jaro_list = jaro(spelling_dict[batch]['trakt'], trakt)
                        block_jaro_list = jaro(spelling_dict[batch]['block'], block)
                        enhet_jaro_list = jaro(spelling_dict[batch]['enhet'], enhet)

                        jaro_trakt = trakt_jaro_list[0:1]
                        jaro_block = block_jaro_list[0:1]
                        jaro_enhet = enhet_jaro_list[0:1]

                        if jaro_trakt[0][1] > 0.7 and jaro_block[0][1] > 0.7 and jaro_enhet[0][1] == 1: #ha jaro_enhet == 1.0 här för att minska false positives

                            sc_pred = (';'.join([jaro_trakt[0][0], jaro_block[0][0], jaro_enhet[0][0]]), (jaro_trakt[0][1], jaro_block[0][1], jaro_enhet[0][1]))
                            for gk in gkomm_list:  
                                try:
                                    df_test  = df_lm.loc[(gk, jaro_trakt[0][0].upper(), jaro_block[0][0].upper(), jaro_enhet[0][0].upper())]
                                except:
                                    continue
                                
                                if len(df_test) == 1:
                                    inst['sc_links'].append((gk + ';' + jaro_trakt[0][0].upper() + ';' + jaro_block[0][0].upper() + ';' + jaro_enhet[0][0].upper(), str(df_test['FNR'].iloc[0])))
                                    no_of_corrected += 1

                        else:
                            sc_pred = ()

                        inst['sc_pred'] = sc_pred

                        

        print('batch: ' + str(batch) + '\tmisses: ' + str(no_of_misses) + '\tcorrected: ' + str(no_of_corrected))
        f.write('batch: ' + str(batch) + '\tmisses: ' + str(no_of_misses) + '\tcorrected: ' + str(no_of_corrected) + '\n')
                   

ipykernel_launcher:45: PerformanceWarning: indexing past lexsort depth may impact performance.
batch: 10001009	misses: 46	corrected: 32
batch: 10001059	misses: 51	corrected: 37
batch: 10001239	misses: 36	corrected: 15
batch: 10001419	misses: 48	corrected: 31
batch: 10001429	misses: 27	corrected: 27
batch: 10001490	misses: 8	corrected: 4
batch: 10001510	misses: 26	corrected: 6
batch: 10001722	misses: 2	corrected: 1
batch: 10001933	misses: 13	corrected: 2
batch: 10002043	misses: 55	corrected: 34
batch: 10002390	misses: 193	corrected: 59
batch: 10002455	misses: 725	corrected: 362
batch: 10003120	misses: 276	corrected: 15
batch: 10003246	misses: 58	corrected: 11
batch: 10003296	misses: 397	corrected: 26
batch: 10003466	misses: 91	corrected: 59
batch: 10003654	misses: 77	corrected: 51
batch: 10003664	misses: 42	corrected: 0
batch: 10003724	misses: 55	corrected: 44
batch: 10003774	misses: 51	corrected: 2
batch: 10003814	misses: 186	corrected: 1
batch: 10003833	misses: 607	corrected: 6
batch:

In [12]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/iteration3_sc_epoch3.json', 'w', encoding='utf8') as f:
    j = json.dumps(sc_dict, indent = 4, ensure_ascii=False)
    f.write(str(j))

In [6]:
gkomm_dict['10002390']

['O-GÖTEBORG', 'O-VÄSTRA FRÖLUNDA']

In [13]:
all = ['abrorren']
s = 'abborren'

l = jaro(all, s)